# Suspicious Login Attack Detector

This notebook implements a complete suspicious-login detection prototype:

1. Generate **3,000 synthetic rejected-login records** using 10 common email accounts.
2. Create chronological behavior features without future-data leakage.
3. Train a logistic-regression attack classifier.
4. Combine model probability with deterministic security rules.
5. Add a `thret` score from `0` to `1` and recommend an action.
6. Evaluate any new rejected-login attempt by calling one method.

> **Production warning:** The included labels and patterns are synthetic. Retrain and calibrate the model with confirmed real security outcomes before production use.


## 1. Environment

Required packages: `pandas`, `numpy`, `scikit-learn`, `joblib`, and `matplotlib`.

Uncomment the installation command only when the packages are missing.


## 2. Imports, schema, features, and thresholds

In [2]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd


## 3. Synthetic data generation

The generator creates normal user mistakes and four attack scenarios:

- Brute force against one account
- Credential stuffing across multiple accounts
- Distributed attacks against one account
- Possible account takeover from a new device, IP, or location

Documentation-only IP ranges are used, so the generated addresses do not represent real customers.


In [3]:
DEFAULT_SECONDS = 2_592_000  # 30 days
df = pd.read_csv(Path('../data/login_reject_history_3000.csv'))
df['time_to_attempt'] = pd.to_datetime(df['time_to_attempt'])


# We have
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 9 columns):

| sl | Column           | Non-Null Count | Dtype                  |
|:--:|:-----------------|:---------------|:-----------------------|
| 0  | id               | 3000 non-null  | int64                  |
| 1  | email            | 3000 non-null  | object                 |
| 2  | device_mac_id    | 3000 non-null  | object                 |
| 3  | ip               | 3000 non-null  | object                 |
| 4  | location         | 3000 non-null  | object                 |
| 5  | time_to_attempt  | 3000 non-null  | datetime64[ns, UTC]    |
| 6  | rejection_reason | 3000 non-null  | object                 |
| 7  | is_suspicious    | 3000 non-null  | int64                  |
| 8  | scenario         | 3000 non-null  | object                 |


dtypes: datetime64[ns, UTC](1), int64(2), object(6)
memory usage: 211.1+ KB
# Missing-Value Rules for 22 Engineered Features

| Feature                              | Missing-value rule                                                    |
| ------------------------------------ | --------------------------------------------------------------------- |
| `seconds_since_email_last_attempt`   | Set to `2592000` seconds — 30 days                                    |
| `email_attempts_5m`                  | `0`                                                                   |
| `email_attempts_1h`                  | `0`                                                                   |
| `email_attempts_24h`                 | `0`                                                                   |
| `consecutive_email_failures`         | `0`                                                                   |
| `seconds_since_ip_last_attempt`      | Set to `2592000`                                                      |
| `ip_attempts_5m`                     | `0`                                                                   |
| `ip_attempts_1h`                     | `0`                                                                   |
| `ip_unique_emails_10m`               | `0`                                                                   |
| `ip_unique_emails_1h`                | `0`                                                                   |
| `device_unique_emails_1h`            | `0`                                                                   |
| `email_unique_ips_1h`                | `0`                                                                   |
| `email_unique_ips_24h`               | `0`                                                                   |
| `email_unique_devices_24h`           | `0`                                                                   |
| `email_unique_locations_24h`         | `0`                                                                   |
| `is_new_ip_for_email`                | `0` when the email has no previous history; otherwise calculate `0/1` |
| `is_new_device_for_email`            | `0` when the email has no previous history; otherwise calculate `0/1` |
| `is_new_location_for_email`          | `0` when the email has no previous history; otherwise calculate `0/1` |
| `location_changed_from_last_attempt` | `0` when no previous attempt exists                                   |
| `seconds_since_device_last_attempt`  | Set to `2592000`                                                      |
| `hour_sin`                           | Recalculate from `time_to_attempt`; do not statistically impute       |
| `hour_cos`                           | Recalculate from `time_to_attempt`; do not statistically impute       |




In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   id                3000 non-null   int64              
 1   email             3000 non-null   object             
 2   device_mac_id     3000 non-null   object             
 3   ip                3000 non-null   object             
 4   location          3000 non-null   object             
 5   time_to_attempt   3000 non-null   datetime64[ns, UTC]
 6   rejection_reason  3000 non-null   object             
 7   is_suspicious     3000 non-null   int64              
 8   scenario          3000 non-null   object             
dtypes: datetime64[ns, UTC](1), int64(2), object(6)
memory usage: 211.1+ KB


In [8]:
df = df.sort_values(['email', 'time_to_attempt'])

email_group = df.groupby('email', sort=False)
df['email_attempts_5m'] = (email_group.rolling(
    window="5min",
    on="time_to_attempt",
    closed="right"
)["id"]
                           .count()
                           .astype(int)
                           .to_numpy()
                           )

df['email_attempts_10m'] = (email_group.rolling(
    window="10min",
    on="time_to_attempt",
    closed="right"
)["id"]
                            .count()
                            .astype(int)
                            .to_numpy()
                            )

df['email_attempts_1h'] = (email_group.rolling(
    window="1h",
    on="time_to_attempt",
    closed="right"
)["id"]
                           .count()
                           .astype(int)
                           .to_numpy()
                           )

df["seconds_since_email_last_attempt"] = (
    email_group["time_to_attempt"]
    .diff()
    .dt.total_seconds()
    .fillna(2592000)
    .astype(int)
)

time_diff = email_group['time_to_attempt'].diff().dt.total_seconds()
df['rapid_attempt_flag'] = (time_diff.lt(60).astype(int).fillna(0))

df['location_change_flag'] = (email_group['location']
                              .transform(lambda x: x.ne(x.shift()))
                              .astype(int)
                              )

df['failed_attempt_streak_email'] = (
    email_group['is_suspicious']
    .transform(
        lambda s: s.groupby(
            s.eq(0).cumsum()
        ).cumsum()
    )
)
df['device_mac_code'] = pd.factorize(df['device_mac_id'])[0]
df['email_unique_devices_1h'] = (email_group
                                 .rolling(window='1h', on='time_to_attempt', min_periods=1, closed='both')[
                                     'device_mac_code']
                                 .apply(lambda x: np.unique(x[x >= 0]).size, raw=True)
                                 .to_numpy()
                                 .astype(int)
                                 )

df['ip_code'] = pd.factorize(df['ip'])[0]
df['email_unique_ips_1h'] = (email_group
                             .rolling(window='1h', on='time_to_attempt', min_periods=1, closed='both')['ip_code']
                             .apply(lambda x: len(np.unique(x)), raw=True)
                             .to_numpy()
                             .astype(int)
                             )

df = df.sort_values(['ip', 'time_to_attempt'])
ip_group = df.groupby('ip')
df['seconds_since_ip_last_attempt'] = (ip_group['time_to_attempt'].diff()
                                       .dt.total_seconds()
                                       .fillna(2592000)
                                       .astype(int))

df['ip_attempts_5m'] = (ip_group.rolling(
    window="5min",
    on="time_to_attempt",
    closed="right"
)["id"]
                        .count()
                        .astype(int)
                        .to_numpy()
                        )

df['ip_attempts_10m'] = (ip_group.rolling(
    window="10min",
    on="time_to_attempt",
    closed="right"
)["id"]
                         .count()
                         .astype(int)
                         .to_numpy()
                         )

df['email_code'] = pd.factorize(df['email'])[0]
df['ip_unique_emails_1h'] = (ip_group
                             .rolling(window='1h', on='time_to_attempt', min_periods=1, closed='both')['email_code']
                             .apply(lambda x: len(np.unique(x)), raw=False)
                             .to_numpy()
                             .astype(int)
                             )

df['failed_attempt_streak_ip'] = (
    ip_group['is_suspicious']
    .transform(
        lambda s: s.groupby(
            s.eq(0).cumsum()
        ).cumsum()
    )
)
hour = df["time_to_attempt"].dt.hour

df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * hour / 24)



df = df.sort_values(['device_mac_id', 'time_to_attempt'])

mac_group = df.groupby('device_mac_id', sort=False)
df['device_attempts_5m'] = (mac_group.rolling(
    window="5min",
    on="time_to_attempt",
    closed="right"
)["id"]
                            .count()
                            .astype(int)
                            .to_numpy()
                            )

failed = df['is_suspicious'].eq(0)
groups = (~failed).groupby(df['device_mac_id']).cumsum()
df['failed_attempt_streak_device'] = (
    failed.groupby([df['device_mac_id'], groups]).cumcount().astype(int)
)

df['device_attempts_1h'] = (mac_group.rolling(
    window="1h",
    on="time_to_attempt",
    closed="right"
)["id"]
                            .count()
                            .astype(int)
                            .to_numpy()
                            )




In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3000 entries, 211 to 1185
Data columns (total 31 columns):
 #   Column                            Non-Null Count  Dtype              
---  ------                            --------------  -----              
 0   id                                3000 non-null   int64              
 1   email                             3000 non-null   object             
 2   device_mac_id                     3000 non-null   object             
 3   ip                                3000 non-null   object             
 4   location                          3000 non-null   object             
 5   time_to_attempt                   3000 non-null   datetime64[ns, UTC]
 6   rejection_reason                  3000 non-null   object             
 7   is_suspicious                     3000 non-null   int64              
 8   scenario                          3000 non-null   object             
 9   email_attempts_5m                 3000 non-null   int64           

In [10]:
# email_group
df.head(10)


,id,email,device_mac_id,ip,location,time_to_attempt,rejection_reason,is_suspicious,scenario,email_attempts_5m,...,ip_attempts_5m,ip_attempts_10m,email_code,ip_unique_emails_1h,failed_attempt_streak_ip,hour_sin,hour_cos,device_attempts_5m,failed_attempt_streak_device,device_attempts_1h
211,212,user2@example.com,02:00:4E:3D:31:7E,198.51.100.27,"Dubai, AE",2026-07-03 14:07:49+00:00,unknown_device,1,distributed_account_attack,2,...,1,1,7,1,1,-0.500000,-0.866025,1,0,1
506,507,user6@example.com,02:00:EC:D3:1B:60,192.0.2.207,"Singapore, SG",2026-07-06 15:03:30+00:00,password_typo,0,normal_user_error,1,...,1,1,5,1,0,-0.707107,-0.707107,1,0,1
691,692,user8@example.com,02:00:FA:7E:F1:38,198.51.100.210,"Dhaka, BD",2026-07-08 11:49:06+00:00,invalid_password,1,distributed_account_attack,1,...,1,1,1,1,1,0.258819,-0.965926,1,0,1
1571,1572,user6@example.com,02:02:51:82:3E:62,198.51.100.124,"Dhaka, BD",2026-07-17 20:01:56+00:00,invalid_password,1,brute_force_same_account,1,...,1,1,5,1,1,-0.866025,0.500000,1,0,1
1572,1573,user6@example.com,02:02:51:82:3E:62,198.51.100.124,"Dhaka, BD",2026-07-17 20:02:16+00:00,too_many_attempts,1,brute_force_same_account,2,...,2,2,5,1,2,-0.866025,0.500000,2,0,2
1573,1574,user6@example.com,02:02:51:82:3E:62,198.51.100.124,"Dhaka, BD",2026-07-17 20:02:17+00:00,invalid_password,1,brute_force_same_account,3,...,3,3,5,1,3,-0.866025,0.500000,3,0,3
1574,1575,user6@example.com,02:02:51:82:3E:62,198.51.100.124,"Dhaka, BD",2026-07-17 20:02:18+00:00,too_many_attempts,1,brute_force_same_account,4,...,4,4,5,1,4,-0.866025,0.500000,4,0,4
1575,1576,user6@example.com,02:02:51:82:3E:62,198.51.100.124,"Dhaka, BD",2026-07-17 20:02:20+00:00,invalid_password,1,brute_force_same_account,5,...,5,5,5,1,5,-0.866025,0.500000,5,0,5
1576,1577,user6@example.com,02:02:51:82:3E:62,198.51.100.124,"Dhaka, BD",2026-07-17 20:02:38+00:00,invalid_password,1,brute_force_same_account,6,...,6,6,5,1,6,-0.866025,0.500000,6,0,6
1577,1578,user6@example.com,02:02:51:82:3E:62,198.51.100.124,"Dhaka, BD",2026-07-17 20:02:41+00:00,invalid_password,1,brute_force_same_account,7,...,7,7,5,1,7,-0.866025,0.500000,7,0,7


In [11]:
# final_df= add_risk_features(df)
# final_df.to_csv('~/featured_login.csv', index=False)
# df.info()
df.to_csv('~/featured_login.csv', index=False)
# df['seconds_since_email_last_attempt'].min()
# df['email_unique_devices_1h'].unique()
# def lambdafun(x):
#     return len(x)
# f=(df[df['email']=='user10@example.com'].sort_values(['email','time_to_attempt','device_mac_id'],kind='stable')
#                             .groupby(['email'],sort=False)
#                            .rolling(window='1h',on='time_to_attempt',min_periods=1)['device_mac_code']
#                             .apply(lambdafun, raw=False)
#                             .reset_index(drop=True)
#                            )
# pd.set_option("display.max_rows", 3000)
# s=pd.DataFrame(f)

# s.head(2000)

In [13]:
df.columns

Index(['id', 'email', 'device_mac_id', 'ip', 'location', 'time_to_attempt',
       'rejection_reason', 'is_suspicious', 'scenario', 'email_attempts_5m',
       'email_attempts_10m', 'email_attempts_1h',
       'seconds_since_email_last_attempt', 'rapid_attempt_flag',
       'location_change_flag', 'failed_attempt_streak_email',
       'device_mac_code', 'email_unique_devices_1h', 'ip_code',
       'email_unique_ips_1h', 'seconds_since_ip_last_attempt',
       'ip_attempts_5m', 'ip_attempts_10m', 'email_code',
       'ip_unique_emails_1h', 'failed_attempt_streak_ip', 'hour_sin',
       'hour_cos', 'device_attempts_5m', 'failed_attempt_streak_device',
       'device_attempts_1h'],
      dtype='object')

# Next
- cleanup irrelevent feature
- feature explaination using proper documentation
- data type checking and pattern validation like false ip , false location
- missing data checking and putting default value 